# test_case_results

用于汇总并展示 `var/runs` 下每一次 case 的执行结果（含 task_id 与 controller trace）。


In [ ]:
from pathlib import Path
import sys
import json
import subprocess
from collections import Counter, defaultdict

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / 'src' / 'task_router_graph').exists()
)
RUNS_DIR = PROJECT_ROOT / 'var' / 'runs'
REPORT_CSV = PROJECT_ROOT / 'var' / 'reports' / 'cases_merged.csv'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('Python =', sys.executable)
print('RUNS_DIR exists =', RUNS_DIR.exists())


In [ ]:
# 开关：按需打开（默认不重跑 case，只读取已有结果）。
RUN_CASES = False
EXPORT_CSV = True
HEARTBEAT_SEC = 10
CASES_DIR = 'data/cases'

print('RUN_CASES =', RUN_CASES)
print('EXPORT_CSV =', EXPORT_CSV)


In [ ]:
if RUN_CASES:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'run_cases.py'),
        '--cases-dir',
        CASES_DIR,
        '--heartbeat-sec',
        str(HEARTBEAT_SEC),
    ]
    print('Running:', ' '.join(cmd))
    proc = subprocess.run(cmd, cwd=PROJECT_ROOT)
    print('run_cases.py return code =', proc.returncode)
else:
    print('Skip run_cases.py (RUN_CASES=False)')

if EXPORT_CSV:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'export_var_cases_csv.py'),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print('Skip export_var_cases_csv.py (EXPORT_CSV=False)')


In [ ]:
def _safe_dict(value):
    return value if isinstance(value, dict) else {}


def _safe_list(value):
    return value if isinstance(value, list) else []


def _collect_rows(runs_dir: Path):
    rows = []

    for run_dir in sorted(path for path in runs_dir.glob('run_*') if path.is_dir()):
        run_token = run_dir.name
        env_path = run_dir / 'environment.json'
        if not env_path.exists():
            continue

        try:
            env_payload = json.loads(env_path.read_text(encoding='utf-8'))
        except Exception:
            rows.append({
                'run_token': run_token,
                'case_id': '',
                'round_id': '',
                'task_id': '',
                'task_type': '',
                'task_status': 'runtime_failed',
                'task_result': f'environment parse failed: {env_path}',
                'reply': '',
                'controller_trace': [],
            })
            continue

        env_payload = _safe_dict(env_payload)
        case_id = str(env_payload.get('case_id', '')).strip()
        rounds = _safe_list(env_payload.get('rounds'))

        if not rounds:
            rows.append({
                'run_token': run_token,
                'case_id': case_id,
                'round_id': '',
                'task_id': '',
                'task_type': '',
                'task_status': 'empty',
                'task_result': 'no rounds found',
                'reply': '',
                'controller_trace': [],
            })
            continue

        for round_item in rounds:
            round_item = _safe_dict(round_item)
            round_id = round_item.get('round_id', '')
            tasks = _safe_list(round_item.get('tasks'))

            if not tasks:
                rows.append({
                    'run_token': run_token,
                    'case_id': case_id,
                    'round_id': round_id,
                    'task_id': '',
                    'task_type': '',
                    'task_status': 'empty',
                    'task_result': 'no tasks found',
                    'reply': '',
                    'controller_trace': [],
                })
                continue

            for task_item in tasks:
                task_item = _safe_dict(task_item)
                task_payload = _safe_dict(task_item.get('task'))
                trace = _safe_list(task_item.get('controller_trace'))

                rows.append({
                    'run_token': run_token,
                    'case_id': case_id,
                    'round_id': round_id,
                    'task_id': task_item.get('task_id', task_payload.get('task_id', '')),
                    'task_type': task_payload.get('type', ''),
                    'task_status': task_payload.get('status', ''),
                    'task_result': task_payload.get('result', ''),
                    'reply': task_item.get('reply', ''),
                    'controller_trace': trace,
                })

    return rows


rows = _collect_rows(RUNS_DIR)
print('rows =', len(rows))

status_counter = Counter((str(r.get('task_status', '')).strip() or 'unknown') for r in rows)
print('status summary =', dict(status_counter))


In [ ]:
def _short(text, limit=140):
    text = str(text).replace('\n', ' ').strip()
    if len(text) <= limit:
        return text
    return text[: limit - 3] + '...'


flat_rows = []
for row in rows:
    flat_rows.append({
        'case_id': row.get('case_id', ''),
        'run_token': row.get('run_token', ''),
        'round_id': row.get('round_id', ''),
        'task_id': row.get('task_id', ''),
        'task_type': row.get('task_type', ''),
        'task_status': row.get('task_status', ''),
        'task_result': _short(row.get('task_result', ''), 100),
        'reply': _short(row.get('reply', ''), 100),
        'trace_count': len(row.get('controller_trace', [])),
    })

flat_rows = sorted(
    flat_rows,
    key=lambda x: (
        str(x.get('case_id', '') or x.get('run_token', '')),
        str(x.get('run_token', '')),
        int(x.get('round_id') or 0),
        int(x.get('task_id') or 0),
    ),
)

try:
    import pandas as pd
    display(pd.DataFrame(flat_rows))
except Exception:
    for item in flat_rows:
        print(item)


In [ ]:
# 如果只看部分 case，填入例如 ['case_01', 'case_02']
CASE_FILTER = []
SHOW_TRACE = True
MAX_TRACE_TEXT = 120


def _short(text, limit=MAX_TRACE_TEXT):
    text = str(text).replace('\n', ' ').strip()
    if len(text) <= limit:
        return text
    return text[: limit - 3] + '...'


grouped = defaultdict(list)
for row in rows:
    case_key = str(row.get('case_id') or row.get('run_token') or 'unknown')
    grouped[case_key].append(row)

case_keys = sorted(grouped.keys())
if CASE_FILTER:
    case_keys = [k for k in case_keys if k in CASE_FILTER]

print('show cases =', len(case_keys))

for case_key in case_keys:
    print('=' * 100)
    print(f'CASE: {case_key}')
    case_rows = sorted(
        grouped[case_key],
        key=lambda x: (str(x.get('run_token', '')), int(x.get('round_id') or 0), int(x.get('task_id') or 0)),
    )

    for row in case_rows:
        print(
            f"- run={row.get('run_token', '')} round={row.get('round_id', '')} "
            f"task_id={row.get('task_id', '')} type={row.get('task_type', '')} status={row.get('task_status', '')}"
        )
        print('  result:', _short(row.get('task_result', ''), 180))
        print('  reply :', _short(row.get('reply', ''), 180))

        trace = row.get('controller_trace', [])
        if SHOW_TRACE and trace:
            print('  trace:')
            for idx, action in enumerate(trace, start=1):
                action = action if isinstance(action, dict) else {}
                print(
                    f"    [{idx}] action={action.get('action_kind', '')} "
                    f"tool={action.get('tool', '')} "
                    f"reason={_short(action.get('reason', ''), 100)} "
                    f"observation={_short(action.get('observation', ''), 100)}"
                )


## 说明

- 该 notebook 默认直接读取 `var/runs` 历史结果；把 `RUN_CASES=True` 后可先重跑。
- 展示以 `environment.json` 内的 `case_id` 为主。
- 失败任务会保留并展示 `controller_trace`，可直接定位 `route_failed` 之类问题。
